# Parquet vs CSV

**John González**

*10 de octubre de 2026*

In [1]:
import kagglehub
from dotenv import load_dotenv

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv("../.env")

# Download latest version
path = kagglehub.competition_download('new-york-city-taxi-fare-prediction')

print("Path to competition files:", path)


Path to competition files: /home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction


In [ ]:
path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

Lectura de datos ansiosa (eager) con polars

In [3]:
! ls -lah $path # Verficamos los archivos descargados

total 7.2G
drwxr-xr-x 2 vscode vscode 4.0K Oct 17 18:13 .
drwxr-xr-x 3 vscode vscode 4.0K Sep 29 02:24 ..
-rw-r--r-- 1 vscode vscode  486 Sep 29 02:22 GCP-Coupons-Instructions.rtf
-rw-r--r-- 1 vscode vscode 336K Sep 29 02:22 sample_submission.csv
-rw-r--r-- 1 vscode vscode 960K Sep 29 02:22 test.csv
-rw-r--r-- 1 vscode vscode 5.4G Sep 29 02:24 train.csv
-rw-r--r-- 1 vscode vscode 1.9G Oct 17 18:15 train.parquet


In [4]:
import time
import polars as pl

inicio = time.time()
df = pl.read_csv(path + "/train.csv")
# Calcular el promedio de la columna "fare_amount"
mean_fare = df.select(pl.col("fare_amount").mean()).item()
print("Promedio de fare_amount: ", mean_fare)
print("Tiempo final de lectura de CSV: ", time.time() - inicio, "segundos")

: 

Leer tabla con polars con el método perezoso (lazy)

In [1]:
import time
import polars as pl

path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

inicio = time.time()

# Escaneo de CSV con Polars, no leemos todo el archivo
df = pl.scan_csv(path + "/train.csv")

# Escanea el promedio de la columna "fare_amount", hace un query plan
q = df.select(pl.col("fare_amount").mean())

# Con collect() ejecutamos la consulta y obtenemos el resultado
mean_fare = q.collect().item()
print("Promedio de fare_amount: ", mean_fare)
print("Tiempo final de lectura de CSV: ", time.time() - inicio, "segundos")

Promedio de fare_amount:  11.345045601662925
Tiempo final de lectura de CSV:  181.79373860359192 segundos


In [2]:
# Veamos que hizo polars internamente

print(q.explain())


SELECT [col("fare_amount").mean()]
  Csv SCAN [/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction/train.csv]
  PROJECT 1/8 COLUMNS
  ESTIMATED ROWS: 56971783


Veamos que ocurre con parquet

In [3]:
import os
import time
import polars as pl

path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

inicio = time.time()

archivo = path + "/train.parquet"

# Transformamos el CSV a Parquet
pl.scan_csv(path + "/train.csv").sink_parquet(archivo)

print("Tiempo final de escritura de Parquet: ", time.time() - inicio, "segundos")

peso_csv = os.path.getsize(path + "/train.csv") / (1024 * 1024)
peso_parquet = os.path.getsize(archivo) / (1024 * 1024)

print("Peso del CSV: ", peso_csv, "MB")
print("Peso del Parquet: ", peso_parquet, "MB")

Tiempo final de escritura de Parquet:  100.99170875549316 segundos
Peso del CSV:  5433.252618789673 MB
Peso del Parquet:  1845.0342712402344 MB


In [4]:
porcentaje_reduccion = (peso_csv - peso_parquet) / peso_csv * 100
print("Porcentaje de reducción: ", porcentaje_reduccion, "%")

Porcentaje de reducción:  66.04180956247826 %


In [5]:
# Leer el esquema del archivo Parquet
esquema = pl.read_parquet_schema(archivo)

print("Esquema del archivo Parquet:")

for columna, tipo in esquema.items():
    print(f"{columna}: {tipo}")
    

Esquema del archivo Parquet:
key: String
fare_amount: Float64
pickup_datetime: String
pickup_longitude: Float64
pickup_latitude: Float64
dropoff_longitude: Float64
dropoff_latitude: Float64
passenger_count: Int64


In [6]:
##Con parquet tengo la ventaja que para conocer las columnas y su tipo no necesito leer toda la tabla, la información hace parte de la metadata propia del archivo, ahorrando minutos y problemas de memoria por lectura innecesaria solo para obtener esta información básica.

In [7]:
# Si solo quiero leer las primeras 5 filas del archivo Parquet, puedo usar el método head() de Polars:
muestra = pl.scan_parquet(archivo).head(5).collect()
print("Muestra de las primeras 5 filas del archivo Parquet:")
print(muestra)

Muestra de las primeras 5 filas del archivo Parquet:
shape: (5, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ key        ┆ fare_amoun ┆ pickup_dat ┆ pickup_lo ┆ pickup_la ┆ dropoff_l ┆ dropoff_l ┆ passenger │
│ ---        ┆ t          ┆ etime      ┆ ngitude   ┆ titude    ┆ ongitude  ┆ atitude   ┆ _count    │
│ str        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ f64        ┆ str        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ i64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2009-06-15 ┆ 4.5        ┆ 2009-06-15 ┆ -73.84431 ┆ 40.721319 ┆ -73.84161 ┆ 40.712278 ┆ 1         │
│ 17:26:21.0 ┆            ┆ 17:26:21   ┆ 1         ┆           ┆           ┆           ┆           │
│ 000001     ┆            ┆ UTC        ┆           ┆           ┆           ┆           ┆           │
│ 2010-01-05 ┆ 16.9     

Lectura a profundidad de un archivo parquet

In [8]:
import pyarrow.parquet as pq

path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

archivo = path + "/train.parquet"

# Para leer la metadata

metadata = pq.read_metadata(archivo)

print("Número de filas: ", metadata.num_rows)
print("Número de columnas: ", metadata.num_columns)
print("Número de grupos de filas: ", metadata.num_row_groups)
print("Filas por grupo: ", metadata.row_group(0).num_rows)
print("Versión de Parquet: ", metadata.format_version)

Número de filas:  55423856
Número de columnas:  8
Número de grupos de filas:  640
Filas por grupo:  83488
Versión de Parquet:  1.0


Un archivo parquet generalmente viene particionado por grupos, en este caso tenemos 640 grupos, cada uno con ~87k filas. La lectura de esta información se hace directamente desde la metadata sin necesidad de leer el archivo completo, haciendo esta lectura básica instantáneamente con la librería pyarrow.


In [9]:
# Detalle del grupo 1 row=0
 
row_group = metadata.row_group(0)
print("Número de filas en el grupo 1: ", row_group.num_rows)
print(f"Tamaño del grupo 1: {row_group.total_byte_size / (1024 * 1024):.2f} MB")
 
print(" Estructura del grupo 1:")
 
for i in range(row_group.num_columns):
    column = row_group.column(i)
    print(f"Columna {i+1}: {column.path_in_schema}")
    print(f"  - Tipo: {column.physical_type}")
    print(f"  - Tamaño: {column.total_compressed_size / (1024 * 1024):.2f} MB")
    print(f"  - Número de valores: {column.num_values}")
    print(f"  - Número de valores nulos: {column.statistics.null_count}")
    print(f"  - Valor mínimo: {column.statistics.min}")
    print(f"  - Valor máximo: {column.statistics.max}")
    print(f"  - Compresión: {column.compression}")

Número de filas en el grupo 1:  83488
Tamaño del grupo 1: 7.89 MB
 Estructura del grupo 1:
Columna 1: key
  - Tipo: BYTE_ARRAY
  - Tamaño: 0.67 MB
  - Número de valores: 83488
  - Número de valores nulos: 0
  - Valor mínimo: 2009-01-01 00:41:00.0000003
  - Valor máximo: 2015-06-30 22:42:39.00000014
  - Compresión: ZSTD
Columna 2: fare_amount
  - Tipo: DOUBLE
  - Tamaño: 0.14 MB
  - Número de valores: 83488
  - Número de valores nulos: 0
  - Valor mínimo: -44.9
  - Valor máximo: 200.0
  - Compresión: ZSTD
Columna 3: pickup_datetime
  - Tipo: BYTE_ARRAY
  - Tamaño: 0.51 MB
  - Número de valores: 83488
  - Número de valores nulos: 0
  - Valor mínimo: 2009-01-01 00:41:00 UTC
  - Valor máximo: 2015-06-30 22:42:39 UTC
  - Compresión: ZSTD
Columna 4: pickup_longitude
  - Tipo: DOUBLE
  - Tamaño: 0.35 MB
  - Número de valores: 83488
  - Número de valores nulos: 0
  - Valor mínimo: -736.55
  - Valor máximo: 40.783472
  - Compresión: ZSTD
Columna 5: pickup_latitude
  - Tipo: DOUBLE
  - Tamaño: 0

Cacular promedio nuevamente usando parquet

In [10]:
import time
import polars as pl

path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

inicio = time.time()

# Escaneo de Parquet con Polars, no leemos todo el archivo
df = pl.scan_parquet(path + "/train.parquet")

# Escanea el promedio de la columna "fare_amount", hace un query plan
q = df.select(pl.col("fare_amount").mean())

# Con collect() ejecutamos la consulta y obtenemos el resultado
mean_fare = q.collect().item()
print("Promedio de fare_amount: ", mean_fare)
print("Tiempo final de lectura de Parquet: ", time.time() - inicio, "segundos")

Promedio de fare_amount:  11.345045601662939
Tiempo final de lectura de Parquet:  5.541417360305786 segundos


In [11]:
tiempo_csv = 295.1692488193512
tiempo_parquet = 6.13890266418457
print(f"Porcentaje de reducción de tiempo: {((tiempo_csv - tiempo_parquet) / tiempo_csv * 100):.0f}%")

Porcentaje de reducción de tiempo: 98%


El tiempo pasó de 295s a 6s. Una optimización del 98%. Se paga muy bien la inversión de los ~2mins convirtiendo el archivo de csv a parquet.

Ahora un query más complejo

In [12]:
import time
import polars as pl

path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

# Vamos a filtrar cuantas carreras tuvieron un pasajero en la zona norte de nueva york

def filtro(tipo_archivo):
    if "csv" in tipo_archivo:
        df = pl.scan_csv(path + "/train.csv")
    elif "parquet" in tipo_archivo:
        df = pl.scan_parquet(path + "/train.parquet")
    else:
        raise ValueError("Tipo de archivo no soportado. Use 'csv' o 'parquet'.")

    q = df.filter(
        (pl.col("passenger_count") == 1) &
        (pl.col("pickup_latitude") > -74)
    )

    # Calcular la cantidad de filas filtradas
    resultado = (
        q                   # Tomo las filas filtradas
        .select(pl.len())   # Calculo la cantidad de filas
        .collect()          # Ejecutamos la consulta (recolecto)
        .item()             # Obtengo el valor como un entero
    )
    return resultado

# PARA PARQUET
inicio = time.time()
num_filas_parquet = filtro("parquet")
print("Número de filas filtradas en Parquet: ", num_filas_parquet)
print("Tiempo final de filtrado en Parquet: ", time.time() - inicio, "segundos")

Número de filas filtradas en Parquet:  38334989
Tiempo final de filtrado en Parquet:  8.098999261856079 segundos


In [13]:
# PARA CSV
inicio = time.time()
num_filas_csv = filtro("csv")
print("Número de filas filtradas en CSV: ", num_filas_csv)
print("Tiempo final de filtrado en CSV: ", time.time() - inicio, "segundos")

Número de filas filtradas en CSV:  38334989
Tiempo final de filtrado en CSV:  32.80246019363403 segundos


Mientras parquet tardó 27s, csv 286s, un rendimiento 10 veces menor con parquet.

In [14]:
!head {path}/train.csv

key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21 UTC,-73.844311,40.721319,-73.84161,40.712278,1
2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16 UTC,-74.016048,40.711303,-73.979268,40.782004,1
2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00 UTC,-73.982738,40.76127,-73.991242,40.750562,2
2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42 UTC,-73.98713,40.733143,-73.991567,40.758092,1
2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00 UTC,-73.968095,40.768008,-73.956655,40.783762,1
2011-01-06 09:50:45.0000002,12.1,2011-01-06 09:50:45 UTC,-74.000964,40.73163,-73.972892,40.758233,1
2012-11-20 20:35:00.0000001,7.5,2012-11-20 20:35:00 UTC,-73.980002,40.751662,-73.973802,40.764842,1
2012-01-04 17:22:00.00000081,16.5,2012-01-04 17:22:00 UTC,-73.9513,40.774138,-73.990095,40.751048,1
2012-12-03 13:10:00.000000125,9,2012-12-03 13:10:00 UTC,-74.006462,40.726713,-73.99

In [15]:
!head {path}/train.parquet

PAR1 ��p��,�� 6 (2015-06-30 22:42:39.000000142009-01-01 01:31:49.0000003   (�/��' \n
�U.�� I:30lFdDFt��M��R�*�[�4����c�]b�G<��9ƚq9�k}I���5�8.�����8�T�A�Y�a�J��%�`��_H�
m����b!b�'�a��<_9@�+۶̔k&m,��*g�/dA�(�'H�pI,d��Ce�6. �aݹə�����"SF�ǁ8�$+�\��]sCL!ɞ����)3Y78y��RA�A��)5��XN��ڀU1V�0c}�6�5�<�fnP�x@�z9�-k�`A�&g�>��h�fv��1Ȗ���u��(�4�aҏ��NB}��8.�Θ
)*DK����Xv�t��I7�z RK�8�4j�z �H��8�I�V����a6�Y�3M��`�2B�+E��:�
-�Y!�f��ģ�y�ɯt�;�X�X�Ju���l���d#�Be<�MS�0~��
R��\��̐�8ñ�$q�a���6��#�O!��-9�Bǈ;\ߌd�M�pA���<��pD�ld{RK�2��=s�G=�'d��'(˸ԃؠ6 S��vԘ��"�\�!�y��V�9��x�
z�J��N�8	q�eg2bH�M�0�!1;ʑEٔ!*Ђɽ�i�@�0S'�C=&z�4��TE�9'	�\�F�N�	���$Zh8K��VQ��^�4������7���O�͐¦2DD�)f��X7�5bgC�W$k��3d��#A3�΃ې1�W|6��aI4���� 9�>�L)ё�H��igx�dB@��Py�<��p�g3+b*��9	g�wek�����Um_v�.*b���)~��胰b~��Ω� ����1�!�ԁ��i1��F�g��Wrw�6�c���r�&|S�u�lSW4�5P�!�>���
�8����W0|Z�Ȇ�����@F��M���wf�!�1�%�89"��a7h4P3��c���

Parquet está escrito en binario principalmente para optimizar espacio de almacenamiento. No es muy cómodo porque no es legible para el humano. Parquet es muy útil cuando trabajamos big data, para lo demás está bien seguir usando csv, excel, json, ...